# A tour of bngsim

[bngsim](https://github.com/lanl/bngsim) simulates biochemical reaction networks: models written in the BioNetGen language (BNGL), SBML, Antimony, or BioNetGen `.net` files, as ODEs or as exact stochastic simulations.

Run each cell with **Shift+Enter**. Your own files are in `/work`.

In [ ]:
import math

import matplotlib.pyplot as plt

import bngsim

print("bngsim", bngsim.__version__)

## A rule-based model

`receptor.bngl` binds a ligand to a receptor with one rule, `L(r) + R(l) <-> L(r!1).R(l!1)`. Loading it runs BioNetGen (BNG2.pl), which turns the rule into a reaction network; bngsim then integrates it.

In [ ]:
model = bngsim.Model.load("/examples/receptor.bngl")
ode = bngsim.Simulator(model, method="ode").run(t_span=(0, 30), n_points=301)

plt.plot(ode.time, ode.observables["Bound"], label="bound")
plt.plot(ode.time, ode.observables["FreeL"], label="free ligand")
plt.xlabel("time (s)")
plt.ylabel("molecules")
plt.legend()
plt.show()

At equilibrium, binding balances unbinding: `kon (L0 - C)(R0 - C) = koff C`. Solving that quadratic gives the exact number bound, to compare with the simulation.

In [ ]:
L0, R0, kon, koff = 1000, 500, 0.001, 0.1
b = kon * (L0 + R0) + koff
exact = (b - math.sqrt(b * b - 4 * kon * kon * L0 * R0)) / (2 * kon)
print(f"bound at equilibrium: exact {exact:.2f}, simulated {ode.observables['Bound'][-1]:.2f}")

## The same model, stochastically

The SSA (Gillespie's algorithm) fires one reaction at a time, with whole molecules. Each seed gives a different path; they scatter around the ODE curve. A model keeps its state between runs, so each run loads it afresh.

In [ ]:
plt.plot(ode.time, ode.observables["Bound"], "k", label="ODE")
for seed in (1, 2, 3):
    ssa = bngsim.Simulator(bngsim.Model.load("/examples/receptor.bngl"), method="ssa").run(
        t_span=(0, 30), n_points=301, seed=seed)
    plt.plot(ssa.time, ssa.observables["Bound"], alpha=0.6, label=f"SSA, seed {seed}")
plt.xlabel("time (s)")
plt.ylabel("molecules bound")
plt.legend()
plt.show()

## An SBML model

`decay.xml` is SBML: `A` decays at rate `k = 0.5`, so `A(t) = 100 exp(-0.5 t)` exactly.

In [ ]:
sbml = bngsim.Simulator(bngsim.Model.load("/examples/decay.xml"), method="ode").run(t_span=(0, 4), n_points=5)
a = sbml.species[:, sbml.species_names.index("A")]
for t, x in zip(sbml.time, a):
    print(f"t = {t:.0f}   A = {x:8.4f}   exact {100 * math.exp(-0.5 * t):8.4f}")

## Your own model

Put a model in the folder you mounted at `/work`, then:

```python
model = bngsim.Model.load("/work/my_model.bngl")
result = bngsim.Simulator(model, method="ode").run(t_span=(0, 100), n_points=201)
result.observables  # one column per observable
```

The [bngsim documentation](https://bngsim.readthedocs.io) covers the rest: sensitivities, steady states, parameter scans and network-free simulation.